# Week 26-1 · TIO-02 — Tackling Desk Operations & the Regulatory Environment

**Instructor:** Nitesh (co-founder/partner, iRage — a private HFT clearing & trading member doing 1-2 million trades/day on Indian exchanges since ~2009).

This is a **business / operations** lecture, not a strategy lecture. So this notebook is a set of **decision
calculators** that reproduce every number the speaker (and the lecture summary + slides) quoted while
explaining how to *set up your own algo-trading desk*: trading paradigms, entity & fund structures, market
access & membership types, co-location (space **and power**), network order-routing and the
leased-line latency-cost curve, the platform latency ladder, the full financial plan, and
operational-risk kill-switches.

All figures are the lecture's own representative numbers (the speaker stressed they are **crude,
illustrative** and not advice). seed `2601`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

np.random.seed(2601)
pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)

# VIOLET / PURPLE theme (matches the cheat sheet & study guide)
VIOLET   = '#6d28d9'
PURPLE   = '#7c3aed'
LILAC    = '#a78bfa'
PLUM     = '#4c1d95'
GREY     = '#6b7280'
AMBER_HL = '#d97706'
print('theme set - seed 2601')

theme set - seed 2601


## Part A · Trading paradigms — defined by *latency sensitivity*

The speaker rejects the textbook LFT/MFT/HFT split and uses a **business** definition tied to how fast your
edge decays (echoed in the summary: *add a few ms and LFT is fine; add a few µs and HFT breaks*):

| Paradigm | Edge survives a delay of… | Edge dies if you add… | Infra implication |
|---|---|---|---|
| **LFT** (low) | ~100 ms or more | (not latency-bound) | no dedicated speed infra |
| **MFT** (mid) | a few ms | a few **100 ms** | maybe co-location |
| **HFT** (high) | a few **µs** | every additional **µs** | co-location is mandatory |

There is **no progression** LFT→MFT→HFT — each is its own business with massive funds in it.

In [2]:
def classify_paradigm(kills_edge_us):
    """Given the added delay (microseconds) at which a strategy's edge collapses,
    return the trading paradigm using the speaker's latency-sensitivity definition."""
    if kills_edge_us >= 100_000:      # 100 ms+
        return 'LFT'
    elif kills_edge_us >= 1_000:      # a few ms
        return 'MFT'
    else:                             # every microsecond matters
        return 'HFT'

examples = pd.DataFrame({
    'strategy': ['weekly factor rotation', 'overnight pairs', 'intraday momentum',
                 'cross-exchange stat-arb', 'order-book imbalance', 'latency arbitrage'],
    'edge_collapses_at_delay': ['~2 s', '~500 ms', '~50 ms', '~2 ms', '~50 us', '~1 us'],
    'kills_edge_us': [2_000_000, 500_000, 50_000, 2_000, 50, 1],
})
examples['paradigm'] = examples['kills_edge_us'].apply(classify_paradigm)
print(examples.to_string(index=False))

               strategy edge_collapses_at_delay  kills_edge_us paradigm
 weekly factor rotation                    ~2 s        2000000      LFT
        overnight pairs                 ~500 ms         500000      LFT
      intraday momentum                  ~50 ms          50000      MFT
cross-exchange stat-arb                   ~2 ms           2000      MFT
   order-book imbalance                  ~50 us             50      HFT
      latency arbitrage                   ~1 us              1      HFT


**Key reframe:** a retail broker API round-trips in *tens of milliseconds* — best case ~30,000 µs. A true
HFT shop lives at **sub-microsecond**. So retail is literally **tens-of-thousands of times slower**, which is
why "HFT" must be used carefully. We quantify that gap in Part F.

In [3]:
retail_api_us = 30_000     # ~30 ms typical broker API round-trip
hft_subus     = 0.5        # sub-microsecond in-house / FPGA
print(f'retail broker API : {retail_api_us:,} us  ({retail_api_us/1000:.0f} ms)')
print(f'HFT sub-us stack  : {hft_subus} us')
print(f'retail is {retail_api_us/hft_subus:,.0f}x slower  -> tens of thousands of times, as stated')

retail broker API : 30,000 us  (30 ms)
HFT sub-us stack  : 0.5 us
retail is 60,000x slower  -> tens of thousands of times, as stated


## Part B · Setting up an entity

Four ways to exist, trading off **speed-to-start** against **liability** and **complexity**:

- **Individual / Proprietor** & **Partnership** — fast, but **unlimited liability** (losses can exceed capital).
- **LLC / Company** — limited liability (the "corporate veil", lifted only on fraud), but heavy compliance;
  needed if you want a **valuation / IPO** (shares exist).
- **LLP** — the global favourite for trading firms: limited liability *plus* partnership flexibility
  (contribution + profit-share, **no shares**). A single founder can use a **One-Person Company (OPC)**.

The lecture summary gives representative company costs: **India ~USD 500 one-time + ~USD 1,000 p.a.**
(2-7 weeks); **Singapore ~USD 1,500 one-time + ~USD 2,000 p.a.** (a few days to 4 weeks).

In [4]:
entities = pd.DataFrame({
    'structure':  ['Individual', 'Partnership', 'LLC / Company', 'LLP', 'OPC (one-person co.)'],
    'liability':  ['Unlimited', 'Unlimited', 'Limited', 'Limited', 'Limited'],
    'shares?':    ['n/a', 'no', 'yes (IPO-able)', 'no (contribution)', 'no'],
    'compliance': ['lowest', 'low', 'highest', 'medium', 'medium'],
    'best_for':   ['retail account', 'few partners, fast', 'valuation / listing',
                   'pro trading firm', 'solo founder wanting limited liability'],
})
print(entities.to_string(index=False))

setup = pd.DataFrame({
    'geography':       ['India', 'Singapore'],
    'time_to_set_up':  ['2-7 weeks', '2 days - 4 weeks'],
    'one_time_usd':    [500, 1_500],
    'annual_usd':      [1_000, 2_000],
})
print()
print(setup.to_string(index=False))

           structure liability           shares? compliance                               best_for
          Individual Unlimited               n/a     lowest                         retail account
         Partnership Unlimited                no        low                     few partners, fast
       LLC / Company   Limited    yes (IPO-able)    highest                    valuation / listing
                 LLP   Limited no (contribution)     medium                       pro trading firm
OPC (one-person co.)   Limited                no     medium solo founder wanting limited liability

geography   time_to_set_up  one_time_usd  annual_usd
    India        2-7 weeks           500        1000
Singapore 2 days - 4 weeks          1500        2000


### Managing *other people's* money — the influence spectrum

From "just give tips" to "pool capital and trade it", cost and regulation climb steeply. Three switches
decide which bucket you fall in: **pooling** (segregated vs pooled), **marketing** (open vs closed-room
only), and **assets** (long-only vs long-short / derivatives).

In [5]:
funds = pd.DataFrame({
    'structure':    ['RA / Investment Advisor', 'Managed account / PMS',
                     'Hedge fund / AIF', 'Mutual fund'],
    'can_trade_for_client?': ['no (tips/advice only)', 'yes (segregated, no pooling)',
                              'yes (pooled)', 'yes (pooled)'],
    'leverage':     ['-', 'no leverage', 'high leverage OK', 'no leverage'],
    'shorting/derivs':['-', 'hedge only', 'long & short, multi-asset', 'no short; derivs=hedge only'],
    'marketing':    ['fee / profit-share', 'no public pooling', 'no public promo (accredited only)', 'YES (billboards)'],
    'min_ticket':   ['-', 'mid', '~$110k IN / $250k US', '~$1'],
})
print(funds.to_string(index=False))
print()
print('IA can take a % of profits but cannot trade the client account. PMS trades but must keep every')
print('client account SEGREGATED (no pooling) and cannot take leverage. Regulators clamp HARDEST on mutual')
print('funds (vulnerable retail public); hedge funds/AIFs are lightly marketed to accredited investors only.')

              structure        can_trade_for_client?         leverage             shorting/derivs                         marketing           min_ticket
RA / Investment Advisor        no (tips/advice only)                -                           -                fee / profit-share                    -
  Managed account / PMS yes (segregated, no pooling)      no leverage                  hedge only                 no public pooling                  mid
       Hedge fund / AIF                 yes (pooled) high leverage OK   long & short, multi-asset no public promo (accredited only) ~$110k IN / $250k US
            Mutual fund                 yes (pooled)      no leverage no short; derivs=hedge only                  YES (billboards)                  ~$1

IA can take a % of profits but cannot trade the client account. PMS trades but must keep every
client account SEGREGATED (no pooling) and cannot take leverage. Regulators clamp HARDEST on mutual
funds (vulnerable retail public); hedge

## Part C · Market access — your own membership vs a broker

Owning the membership buys **low latency, control, IP protection, capital efficiency** and removes
**counter-party / credit risk** (the speaker's own clearer once neared insolvency, so iRage took its own
clearing membership). The cost is a **membership fee + heavy compliance**. Broker access is the reverse:
no network/membership/compliance burden, but you carry **credit risk** (the broker could default) and
inherit the broker's **data quality / latency**.

In [6]:
own_capital = 1_000_000           # $1M of your own cash
bank_guarantee = 1_000_000        # $1M BG (where permitted)
margin_as_client = own_capital                         # cash only
margin_as_member = own_capital + bank_guarantee        # cash + BG / bonds / FDs / G-Secs
leverage = margin_as_member / margin_as_client
print('CAPITAL EFFICIENCY - the bank-guarantee / collateral multiplier')
print(f'  client margin : ${margin_as_client:,.0f}  (cash only)')
print(f'  member margin : ${margin_as_member:,.0f}  (cash + bank guarantee / bonds / FDs)')
print(f'  multiplier    : {leverage:.1f}x  -> ~2x return on capital')
fd_yield = 0.06
print(f'  FD-as-margin also earns ~{fd_yield:.0%}/yr on parked cash = extra alpha a pure client forgoes')

CAPITAL EFFICIENCY - the bank-guarantee / collateral multiplier
  client margin : $1,000,000  (cash only)
  member margin : $2,000,000  (cash + bank guarantee / bonds / FDs)
  multiplier    : 2.0x  -> ~2x return on capital
  FD-as-margin also earns ~6%/yr on parked cash = extra alpha a pure client forgoes


### The four membership types (India)

Market access as a **member** comes in tiers — how much you can do (trade / take clients / clear) rises
with the compliance burden.

In [7]:
membership = pd.DataFrame({
    'type': ['Trading (Prop / Alpha)', 'Trading member', 'Clearing member', 'PCM (Professional CM)'],
    'can_trade?':    ['yes (own account)', 'yes', 'yes', 'no'],
    'can_take_clients?': ['no', 'yes', 'yes', 'clearing clients only'],
    'can_clear?':    ['via a clearing member', 'via a clearing member', 'yes (self + others)', 'yes (only clears)'],
    'compliance':    ['lowest', 'medium', 'high', 'high'],
})
print(membership.to_string(index=False))
print()
print('Rule of thumb (summary): take your OWN membership only when your transaction-cost SAVINGS are')
print('roughly 2-3x the membership + compliance cost - otherwise a broker is simpler at your volume.')

                  type        can_trade?     can_take_clients?            can_clear? compliance
Trading (Prop / Alpha) yes (own account)                    no via a clearing member     lowest
        Trading member               yes                   yes via a clearing member     medium
       Clearing member               yes                   yes   yes (self + others)       high
 PCM (Professional CM)                no clearing clients only     yes (only clears)       high

Rule of thumb (summary): take your OWN membership only when your transaction-cost SAVINGS are
roughly 2-3x the membership + compliance cost - otherwise a broker is simpler at your volume.


### When does buying your own membership pay off? — the cost crossover

A broker charges per volume (no fixed fee); a member pays a big **fixed** fee but a tiny marginal cost.
Below the crossover volume the broker is cheaper; above it, own membership wins.

In [8]:
member_fixed_annual = 120_000     # membership + compliance + lines, fixed
member_per_mn       = 2.0         # $/million orders (self-cleared, cheap)
broker_per_mn       = 60.0        # $/million orders (broker mark-up)

vols_mn = np.arange(0, 6001, 100)             # millions of orders/yr
member_cost = member_fixed_annual + member_per_mn * vols_mn
broker_cost = broker_per_mn * vols_mn
crossover = member_fixed_annual / (broker_per_mn - member_per_mn)   # break-even volume (mn orders)
print(f'break-even volume: {crossover:,.0f} million orders/yr')
print('  below -> broker is cheaper; above -> own membership wins')
print(f'a mid-tier HFT firm sends 10M-100M orders/DAY -> ~{30*1000:,}M+/yr, far past break-even')

break-even volume: 2,069 million orders/yr
  below -> broker is cheaper; above -> own membership wins
a mid-tier HFT firm sends 10M-100M orders/DAY -> ~30,000M+/yr, far past break-even


## Part D · Co-location — fair, finite, space- *and power*-bound

Co-location = your server on the **same network** as the matching engine (merely the *same building* but a
different network is only **proximity hosting**). Fairness is enforced by giving **every rack an equal length
of cable** — distance in the hall is irrelevant, cable length is everything. In fibre, light travels
~**2×10⁸ m/s** → ~**5 ns per metre** (≈ **1 µs per 200 m**). Co-location is **not allowed for commodity
derivatives** (MCX has no colo, only a data centre next door), and Indian exchanges **bar internet inside
colo** (point-to-point only). For HFT, colo is **not an advantage — it's a necessity**.

In [9]:
c_fibre = 2e8            # m/s, speed of signal in optical fibre
hall_len = 200.0          # metres - a big colo hall
lat_per_m_ns = 1e9 / c_fibre
edge_ns = hall_len * lat_per_m_ns
print(f'fibre signal speed : {c_fibre:.0e} m/s  ->  {lat_per_m_ns:.1f} ns per metre')
print(f'a {hall_len:.0f} m cable difference = {edge_ns:.0f} ns = {edge_ns/1000:.1f} us edge  (why equal cable matters)')

fibre signal speed : 2e+08 m/s  ->  5.0 ns per metre
a 200 m cable difference = 1000 ns = 1.0 us edge  (why equal cable matters)


### The rack is **power-bound**, not space-bound

A standard rack is **42U**, but you can never fill it with servers because **power** (and cooling) runs out
first. The summary's worked example: a colo rack ships with a default **6 kWh** power budget and a typical
server draws **~0.5 kWh**, so you fit only about **12 servers** — and you should keep a cushion below that.

In [10]:
U_per_rack   = 42
rack_power_kwh = 6.0        # default power budget for a colo rack
server_kwh     = 0.5        # a typical trading server
max_servers_power = int(rack_power_kwh / server_kwh)
print(f'rack size            : {U_per_rack}U (physical space)')
print(f'rack power budget     : {rack_power_kwh} kWh')
print(f'server draw           : {server_kwh} kWh each')
print(f'servers by POWER      : {max_servers_power}  <- the real limit (keep a cushion, so ~10)')
print(f'servers by SPACE (1U) : {U_per_rack}  <- never reached; power binds first')

# cost geography
cme_colo_month = 15_000
cme_colo_year  = cme_colo_month * 12
india_colo_year = 15_000            # '< $15k/year'
print(f'\nCME full rack   : ${cme_colo_month:,}/mo = ${cme_colo_year:,}/yr')
print(f'India full rack : < ${india_colo_year:,}/yr  -> CME ~{cme_colo_year/india_colo_year:.0f}x costlier')

rack size            : 42U (physical space)
rack power budget     : 6.0 kWh
server draw           : 0.5 kWh each
servers by POWER      : 12  <- the real limit (keep a cushion, so ~10)
servers by SPACE (1U) : 42  <- never reached; power binds first

CME full rack   : $15,000/mo = $180,000/yr
India full rack : < $15,000/yr  -> CME ~12x costlier


## Part E · Network — market data, order-routing budget, and the price of a millisecond

Two data types: **snapshot** (every n periods) and **tick-by-tick (TBT)**. Volumes are large — NSE lists
instruments in the **10,000s**, and one day of TBT is a **few hundred GB on CME** vs **~30-35 GB on NSE**.
Order-routing is throttled by **messages/second** (new + cancel + modify): NSE offers lines of
**40 / 100 / 200 / 400 / 1000** msgs/s; CME allows **1,500 msgs / 3-second window** (= 500/s).

In [11]:
# how many order-routing lines does a mid-tier HFT need?
cme_per_s = 1500 / 3       # 500 msgs/s per CME line
nse_per_s = 1000           # 1000 msgs/s per NSE line
orders_per_day = 100_000_000          # 100M orders/day (mid-tier HFT)
trading_seconds = 6.5 * 3600          # ~6.5h active session
peak_load = orders_per_day / trading_seconds
print(f'avg message load  : {peak_load:,.0f} msgs/s (100M orders / {trading_seconds/3600:.1f}h)')
print(f'lines @ CME (500/s): {np.ceil(peak_load/cme_per_s):.0f}')
print(f'lines @ NSE (1000/s): {np.ceil(peak_load/nse_per_s):.0f}')

# data volumes
print(f'\nTBT one-day data: CME ~few 100 GB  vs  NSE ~30-35 GB   (NSE instruments in the 10,000s)')

avg message load  : 4,274 msgs/s (100M orders / 6.5h)
lines @ CME (500/s): 9
lines @ NSE (1000/s): 5

TBT one-day data: CME ~few 100 GB  vs  NSE ~30-35 GB   (NSE instruments in the 10,000s)


### The price of a millisecond — the leased-line latency-cost curve

The clearest lesson in the slides: **speed costs exponentially**. These are the lecture's representative
leased-line / connectivity tiers — each faster line shaves only a **few ms** but costs a **multiple** more.
The cost **per millisecond saved** explodes as you approach the exchange.

In [12]:
lines = pd.DataFrame({
    'line_usd_per_month': [2_000, 5_000, 45_000, 125_000],
    'latency_ms':         [82, 78, 71, 68],
})
lines['ms_saved_vs_slowest'] = lines['latency_ms'].iloc[0] - lines['latency_ms']
lines['cost_per_ms_saved'] = np.where(
    lines['ms_saved_vs_slowest'] > 0,
    (lines['line_usd_per_month'] - lines['line_usd_per_month'].iloc[0]) / lines['ms_saved_vs_slowest'],
    0.0).round(0)
print(lines.to_string(index=False))
print()
step = lines.diff().dropna()
for i in range(1, len(lines)):
    dms = lines['latency_ms'].iloc[i-1] - lines['latency_ms'].iloc[i]
    dcost = lines['line_usd_per_month'].iloc[i] - lines['line_usd_per_month'].iloc[i-1]
    print(f'  {lines.latency_ms.iloc[i-1]}ms -> {lines.latency_ms.iloc[i]}ms  (save {dms}ms) costs +${dcost:,}/mo = ${dcost/dms:,.0f} per ms')
print('\nGoing 82ms -> 68ms (just 14ms) jumps the line from $2k to $125k/mo. That is why only latency-')
print('sensitive HFT pays for the last millisecond; everyone else stays on a cheap line.')

 line_usd_per_month  latency_ms  ms_saved_vs_slowest  cost_per_ms_saved
               2000          82                    0                0.0
               5000          78                    4              750.0
              45000          71                   11             3909.0
             125000          68                   14             8786.0

  82ms -> 78ms  (save 4ms) costs +$3,000/mo = $750 per ms
  78ms -> 71ms  (save 7ms) costs +$40,000/mo = $5,714 per ms
  71ms -> 68ms  (save 3ms) costs +$80,000/mo = $26,667 per ms

Going 82ms -> 68ms (just 14ms) jumps the line from $2k to $125k/mo. That is why only latency-
sensitive HFT pays for the last millisecond; everyone else stays on a cheap line.


## Part F · The algo-platform latency ladder

Speed costs money exponentially here too. From a retail broker API down to FPGA, latency falls ~5 orders of
magnitude — and below single-digit microseconds you must **strip the OS** and code the strategy **onto the
hardware (FPGA / ASIC)**, bypassing the operating system.

In [13]:
ladder = pd.DataFrame({
    'platform':   ['Broker API (retail)', 'MFT vendor', 'HFT vendor', 'In-house HFT', 'FPGA / ASIC'],
    'latency_us': [30_000.0, 3_000.0, 50.0, 5.0, 0.5],
    'data_feed_delay': ['few 100 ms', '~100 us', '10-100 us', 'few us', '< 1 us'],
    'cost_per_month_usd': ['free-$10', 'few $100', '$5k-$25k', '$100k-$2M', '$100k-$2M'],
})
ladder['x_faster_than_broker'] = (ladder['latency_us'].iloc[0] / ladder['latency_us']).round(0).astype(int)
print(ladder.to_string(index=False))
print(f"\nbroker -> FPGA speed-up: {30000/0.5:,.0f}x  (the 'tens of thousands' gap, quantified)")

           platform  latency_us data_feed_delay cost_per_month_usd  x_faster_than_broker
Broker API (retail)     30000.0      few 100 ms           free-$10                     1
         MFT vendor      3000.0         ~100 us           few $100                    10
         HFT vendor        50.0       10-100 us           $5k-$25k                   600
       In-house HFT         5.0          few us          $100k-$2M                  6000
        FPGA / ASIC         0.5          < 1 us          $100k-$2M                 60000

broker -> FPGA speed-up: 60,000x  (the 'tens of thousands' gap, quantified)


## Part G · The financial plan — how much to *set up* (not trade)

The speaker's **crude, illustrative** capex to *stand up the business* for its first year (people assumed
**$0** — you partner instead of hire). Two end-points: **HFT + own membership in a developed market**
(~quarter-million) vs **MFT, no membership, emerging market** (~$25-30k). Membership/colo/market-data
line-items only apply to the heavier setup.

In [14]:
plan = pd.DataFrame({
    'line_item': ['Entity setup + 1yr running', 'Office rent (suburb, ~5 ppl)',
                  'Market access (membership/lease)', 'Co-location', 'Market data (black-box)',
                  'Network / order-routing lines', 'Hardware + platform', 'People (partnered)'],
    'HFT_dev_membership': [30_000, 24_000, 35_000, 90_000, 18_000, 25_000, 38_000, 0],
    'MFT_emerging_broker': [3_000, 6_000, 0, 0, 1_000, 1_500, 13_000, 0],
})
plan.loc[len(plan)] = ['TOTAL (setup capex)', plan['HFT_dev_membership'].sum(), plan['MFT_emerging_broker'].sum()]
print(plan.to_string(index=False))
print(f"\nHFT+membership (developed) ~ ${plan['HFT_dev_membership'][:-1].sum():,} (~quarter-million)")
print(f"MFT, no membership (emerging) ~ ${plan['MFT_emerging_broker'][:-1].sum():,} (~$25-30k)")
print('NB: this is SETUP capex only - NOT trading capital / AUM. Full HFT needs a few more bundles.')

                       line_item  HFT_dev_membership  MFT_emerging_broker
      Entity setup + 1yr running               30000                 3000
    Office rent (suburb, ~5 ppl)               24000                 6000
Market access (membership/lease)               35000                    0
                     Co-location               90000                    0
         Market data (black-box)               18000                 1000
   Network / order-routing lines               25000                 1500
             Hardware + platform               38000                13000
              People (partnered)                   0                    0
             TOTAL (setup capex)              260000                24500

HFT+membership (developed) ~ $260,000 (~quarter-million)
MFT, no membership (emerging) ~ $24,500 (~$25-30k)
NB: this is SETUP capex only - NOT trading capital / AUM. Full HFT needs a few more bundles.


## Part H · Regulatory, compliance & operational risk

If you trade through a **broker**, the broker carries most compliance. As a **member** you own it:
**conformance testing** (e.g. CME CERT environment via VPN/P2P) *or* **strategy empanelment**
(India — *every automated strategy* approved, and approval is taken by the **broker, not the individual**),
plus **audit / transaction / order / trade logs** retained for years with **multiple backup layers** and
saved control parameters. *These rules bind members, not their clients.*

The biggest automation risk is **operational**, not market: e.g. a cut fibre freezes market data while your
engine keeps firing on **stale prices** — a fast road to bankruptcy. Defences below.

In [15]:
def safe_to_trade(ticks_since_update, max_stale=50, socket_up=True, heartbeat_ok=True):
    """Three independent guards: strategy-level staleness, socket health, exchange heartbeat.
    If the exchange sees the heartbeat drop it can fire Cancel-on-Disconnect (COD)."""
    return socket_up and heartbeat_ok and (ticks_since_update < max_stale)

scenarios = [
    dict(label='normal',             ticks_since_update=2,   socket_up=True,  heartbeat_ok=True),
    dict(label='fibre cut (stale)',  ticks_since_update=500, socket_up=True,  heartbeat_ok=True),
    dict(label='socket dropped',     ticks_since_update=2,   socket_up=False, heartbeat_ok=True),
    dict(label='heartbeat lost->COD',ticks_since_update=2,   socket_up=True,  heartbeat_ok=False),
]
for s in scenarios:
    ok = safe_to_trade(s['ticks_since_update'], socket_up=s['socket_up'], heartbeat_ok=s['heartbeat_ok'])
    print(f"{s['label']:<22} -> {'TRADE' if ok else 'HALT / cancel orders'}")
print(f'\nUS Pattern-Day-Trader rule: >4 intraday equity trades / 5 days -> minimum ${25_000:,} account margin.')

normal                 -> TRADE
fibre cut (stale)      -> HALT / cancel orders
socket dropped         -> HALT / cancel orders
heartbeat lost->COD    -> HALT / cancel orders

US Pattern-Day-Trader rule: >4 intraday equity trades / 5 days -> minimum $25,000 account margin.


## Part I · The picture — four numbers that decide your build

One figure ties the lecture together: the platform latency ladder that prices speed, the **leased-line
latency-cost curve** (the price of a millisecond), the setup capex by ambition, and the membership-vs-broker
cost crossover.

In [16]:
fig, ax = plt.subplots(2, 2, figsize=(13.5, 9.4))
fig.suptitle('Tackling Desk Operations - four numbers that decide your build (TIO-02)',
             fontsize=14, fontweight='bold', color=PLUM)

# Panel 1: leased-line latency-cost curve (the price of a millisecond)
ax1 = ax[0, 0]
ax1.plot(lines['latency_ms'], lines['line_usd_per_month']/1000, 'o-', color=VIOLET, lw=2, ms=8)
for _, r in lines.iterrows():
    ax1.annotate(f"${r.line_usd_per_month/1000:.0f}k", (r.latency_ms, r.line_usd_per_month/1000),
                 textcoords='offset points', xytext=(6, 6), fontsize=8, color=PLUM)
ax1.invert_xaxis()   # faster (lower latency) to the right
ax1.set_xlabel('connectivity latency (ms) - faster to the right')
ax1.set_ylabel('leased line ($k / month)')
ax1.set_title('E · the price of a millisecond', fontsize=10, color=PLUM)
ax1.grid(alpha=0.3)

# Panel 2: platform latency ladder (log)
ax2 = ax[0, 1]
labels = ladder['platform'][::-1]
vals = ladder['latency_us'][::-1]
bars = ax2.barh(labels, vals, color=['#c4b5fd', LILAC, '#8b5cf6', PURPLE, VIOLET])
ax2.set_xscale('log')
ax2.set_xlabel('round-trip latency (us, log)')
ax2.set_title('F · the platform latency ladder', fontsize=10, color=PLUM)
for b, v in zip(bars, vals):
    ax2.text(v*1.4, b.get_y()+b.get_height()/2, f'{v:g}us', va='center', fontsize=8)
ax2.grid(axis='x', alpha=0.3)

# Panel 3: setup capex stacked
ax3 = ax[1, 0]
items = plan['line_item'][:-1]
hft = plan['HFT_dev_membership'][:-1].values
mft = plan['MFT_emerging_broker'][:-1].values
cmap = plt.cm.Purples(np.linspace(0.35, 0.9, len(items)))
botH = botM = 0
for i, it in enumerate(items):
    ax3.bar('HFT+member\n(developed)', hft[i], bottom=botH, color=cmap[i], edgecolor='white', label=it)
    ax3.bar('MFT, broker\n(emerging)', mft[i], bottom=botM, color=cmap[i], edgecolor='white')
    botH += hft[i]; botM += mft[i]
ax3.set_ylabel('setup capex (USD)')
ax3.set_title(f'G · setup capex  (~{hft.sum()/1000:.0f}k vs ~{mft.sum()/1000:.0f}k USD)', fontsize=10, color=PLUM)
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax3.legend(fontsize=6, ncol=2, loc='upper right')

# Panel 4: membership vs broker cost crossover
ax4 = ax[1, 1]
ax4.plot(vols_mn, broker_cost/1000, color=AMBER_HL, lw=2, label='through broker (per-volume)')
ax4.plot(vols_mn, member_cost/1000, color=VIOLET, lw=2, label='own membership (fixed + tiny)')
ax4.axvline(crossover, color=GREY, ls='--', lw=1)
ax4.text(crossover, ax4.get_ylim()[1]*0.5, f'  break-even\n  {crossover:,.0f}M orders/yr',
         fontsize=8, color=GREY)
ax4.set_xlabel('order volume (millions / yr)')
ax4.set_ylabel('annual access cost ($k)')
ax4.set_title('C · membership vs broker crossover', fontsize=10, color=PLUM)
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('chart_1_deskops.png', dpi=110, bbox_inches='tight')
plt.close()
print('saved chart_1_deskops.png')

print('\n================ VERIFIED DESK-OPS NUMBERS ================')
print(f'retail API vs FPGA        : {retail_api_us/hft_subus:,.0f}x slower')
print(f'capital efficiency (BG)   : {leverage:.1f}x margin')
print(f'membership break-even     : {crossover:,.0f}M orders/yr')
print(f'colo fairness             : {edge_ns/1000:.1f}us edge per {hall_len:.0f}m -> equal cable')
print(f'colo rack power limit     : {max_servers_power} servers ({rack_power_kwh}kWh / {server_kwh}kWh)')
print(f'CME vs India colo         : {cme_colo_year/india_colo_year:.0f}x')
print(f'price of a ms (68ms line) : ${lines.cost_per_ms_saved.iloc[-1]:,.0f}/ms saved')
print(f'order load                : {peak_load:,.0f} msgs/s -> {np.ceil(peak_load/nse_per_s):.0f} NSE lines')
print(f'broker->FPGA ladder       : {30000/0.5:,.0f}x')
print(f'HFT setup capex           : ${hft.sum():,}   MFT setup capex: ${mft.sum():,}')
print('ERRORS-CHECK: notebook ran end-to-end')

saved chart_1_deskops.png

================ VERIFIED DESK-OPS NUMBERS ================
retail API vs FPGA        : 60,000x slower
capital efficiency (BG)   : 2.0x margin
membership break-even     : 2,069M orders/yr
colo fairness             : 1.0us edge per 200m -> equal cable
colo rack power limit     : 12 servers (6.0kWh / 0.5kWh)
CME vs India colo         : 12x
price of a ms (68ms line) : $8,786/ms saved
order load                : 4,274 msgs/s -> 5 NSE lines
broker->FPGA ladder       : 60,000x
HFT setup capex           : $260,000   MFT setup capex: $24,500
ERRORS-CHECK: notebook ran end-to-end
